
# End-to-End Predictive Maintenance using Machine Learning
**Goal:** Predict machine failures and specific failure types using machine learning, and explain the predictions.


In [11]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


## 1. Data Loading and Exploration


In [12]:

# Load the dataset
df = pd.read_csv('dataset/ai4i2020.csv')
df.head()


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [13]:

# Drop useless identifiers
df = df.drop(columns=['UDI', 'Product ID'])

# Create a single 'Failure Type' column for multi-class prediction
def get_failure_type(row):
    if row['TWF'] == 1: return 'TWF'
    if row['HDF'] == 1: return 'HDF'
    if row['PWF'] == 1: return 'PWF'
    if row['OSF'] == 1: return 'OSF'
    if row['RNF'] == 1: return 'RNF'
    return 'None'

df['Failure Type'] = df.apply(get_failure_type, axis=1)



## 2. Feature Engineering
We create new domain-specific features that might help the model learn better.


In [14]:

# 1. Temperature Difference
df['Temperature Difference'] = df['Process temperature [K]'] - df['Air temperature [K]']

# 2. Mechanical Power (Torque * Speed)
df['Mechanical Power'] = df['Rotational speed [rpm]'] * df['Torque [Nm]']

# 3. Wear Ratio
df['Wear Ratio'] = df['Tool wear [min]'] / (df['Rotational speed [rpm]'] + 1e-5)

# 4. Torque-Speed Ratio
df['Torque-Speed Ratio'] = df['Torque [Nm]'] / (df['Rotational speed [rpm]'] + 1e-5)

# 5. Normalized Tool Wear (assuming max wear is ~253 based on data)
df['Normalized Tool Wear'] = df['Tool wear [min]'] / 253.0



## 3. Data Splitting & Pipeline Setup
Instead of scaling and encoding manually step-by-step, we use Scikit-Learn's `Pipeline` and `ColumnTransformer`. 
This is a standard industry practice because it prevents data leakage and makes deploying the model much simpler.


In [15]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Failure Type'])
y_bin = df['Machine failure']
y_type = df['Failure Type']

# Split the data BEFORE preprocessing (Best Practice)
X_train, X_test, y_train_bin, y_test_bin, y_train_type, y_test_type = train_test_split(X, y_bin, y_type, test_size=0.2, random_state=42, stratify=y_bin)

# Define which columns are numeric and which are categorical
categorical_cols = ['Type']
numeric_cols = [col for col in X.columns if col != 'Type']

# Create a preprocessor that handles scaling and encoding automatically
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)



## 4. Model Training (Pipelines)
We bundle the preprocessor and the machine learning model into a single `Pipeline`.


In [16]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# 1. Binary Classification Pipeline (Predicts if failure happens or not)
# We use GridSearchCV to find the absolute best hyperparameters for our pipeline.
binary_pipeline_base = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss'))
])

xgb_params = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.1, 0.2]
}

grid_xgb = GridSearchCV(binary_pipeline_base, xgb_params, cv=3, scoring='f1')
grid_xgb.fit(X_train, y_train_bin)

# Extract the best pipeline found by GridSearchCV
binary_pipeline = grid_xgb.best_estimator_
print("Best XGBoost Parameters:", grid_xgb.best_params_)
print("Binary classification pipeline trained successfully.\n")

# 2. Failure Type Pipeline (Predicts exactly what failed)
# We only train this model on data where a failure actually occurred!
fail_indices = (y_train_bin == 1)
X_train_fail = X_train[fail_indices]
y_train_fail = y_train_type[fail_indices]

type_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# Train the multi-class pipeline
type_pipeline.fit(X_train_fail, y_train_fail)
print("Failure type pipeline trained successfully.")


Best XGBoost Parameters: {'classifier__learning_rate': 0.2, 'classifier__max_depth': 3, 'classifier__n_estimators': 100}
Binary classification pipeline trained successfully.

Failure type pipeline trained successfully.



## 5. Model Evaluation
Let's see how our binary pipeline performs on the test data.


In [17]:

from sklearn.metrics import classification_report, confusion_matrix

# Predict using the pipeline
# Notice how we just pass X_test directly! The pipeline handles the scaling automatically.
preds = binary_pipeline.predict(X_test)

print("Confusion Matrix for XGBoost Pipeline:")
print(confusion_matrix(y_test_bin, preds))

print("\nClassification Report for XGBoost Pipeline:")
print(classification_report(y_test_bin, preds))


Confusion Matrix for XGBoost Pipeline:
[[1929    3]
 [  18   50]]

Classification Report for XGBoost Pipeline:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.94      0.74      0.83        68

    accuracy                           0.99      2000
   macro avg       0.97      0.87      0.91      2000
weighted avg       0.99      0.99      0.99      2000




## 6. Model Saving
Now we save our complete pipelines. We no longer need to save the scaler and encoder separately!


In [18]:

import os
os.makedirs('models', exist_ok=True)

joblib.dump(binary_pipeline, 'models/binary_pipeline.pkl')
joblib.dump(type_pipeline, 'models/failure_type_pipeline.pkl')
print("Pipelines saved successfully!")


Pipelines saved successfully!



## 7. Backend Testing (Streamlit Simulation)
Let's simulate how the backend will process raw user input using our new pipeline.


In [19]:

test_records = [
    {'Scenario': 'No Failure', 'Type': 'M', 'Air temperature [K]': 298.1, 'Process temperature [K]': 308.6, 'Rotational speed [rpm]': 1551, 'Torque [Nm]': 42.8, 'Tool wear [min]': 0},
    {'Scenario': 'TWF (Tool Wear)', 'Type': 'L', 'Air temperature [K]': 298.8, 'Process temperature [K]': 308.9, 'Rotational speed [rpm]': 1455, 'Torque [Nm]': 41.3, 'Tool wear [min]': 208},
    {'Scenario': 'HDF (Heat Dissipation)', 'Type': 'M', 'Air temperature [K]': 300.8, 'Process temperature [K]': 309.4, 'Rotational speed [rpm]': 1342, 'Torque [Nm]': 62.4, 'Tool wear [min]': 113},
    {'Scenario': 'PWF (Power Failure)', 'Type': 'L', 'Air temperature [K]': 298.9, 'Process temperature [K]': 309.1, 'Rotational speed [rpm]': 2861, 'Torque [Nm]': 4.6, 'Tool wear [min]': 143},
    {'Scenario': 'OSF (Overstrain)', 'Type': 'L', 'Air temperature [K]': 298.9, 'Process temperature [K]': 309.0, 'Rotational speed [rpm]': 1410, 'Torque [Nm]': 65.7, 'Tool wear [min]': 191},
    {'Scenario': 'RNF (Random Failure)', 'Type': 'M', 'Air temperature [K]': 297.0, 'Process temperature [K]': 308.3, 'Rotational speed [rpm]': 1399, 'Torque [Nm]': 46.4, 'Tool wear [min]': 132}
]

test_df = pd.DataFrame(test_records)
scenarios = test_df['Scenario'].tolist()
test_df = test_df.drop(columns=['Scenario'])

# Feature Engineering
test_df['Temperature Difference'] = test_df['Process temperature [K]'] - test_df['Air temperature [K]']
test_df['Mechanical Power'] = test_df['Rotational speed [rpm]'] * test_df['Torque [Nm]']
test_df['Wear Ratio'] = test_df['Tool wear [min]'] / (test_df['Rotational speed [rpm]'] + 1e-5)
test_df['Torque-Speed Ratio'] = test_df['Torque [Nm]'] / (test_df['Rotational speed [rpm]'] + 1e-5)
test_df['Normalized Tool Wear'] = test_df['Tool wear [min]'] / 253.0

# Watch how clean this is! Just pass the raw dataframe into the pipeline.
binary_preds = binary_pipeline.predict(test_df)
type_preds = type_pipeline.predict(test_df)

results = []
for i in range(len(scenarios)):
    fail_status = "Failure" if binary_preds[i] == 1 else "Normal"
    fail_type = type_preds[i] if binary_preds[i] == 1 else "N/A"
    results.append({
        "Test Scenario": scenarios[i],
        "Predicted Status": fail_status,
        "Predicted Failure Type": fail_type
    })

display(pd.DataFrame(results))


,Test Scenario,Predicted Status,Predicted Failure Type
0,No Failure,Normal,N/A
1,TWF (Tool Wear),Normal,N/A
2,HDF (Heat Dissipation),Failure,HDF
3,PWF (Power Failure),Failure,PWF
4,OSF (Overstrain),Failure,PWF
5,RNF (Random Failure),Normal,N/A
